# Which Handling Tasks Appear in Injury Narratives: Nursing Facilities and Hospitals

What kind of patient movement was under way when a handling injury occurred: a lateral transfer between two surfaces, repositioning in bed, a seated transfer, recovering a person from the floor, or catching a person who was falling?

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC)
**Data** OSHA ITA Case Detail Data, 2025 (submissions through 2026-08-31)
**Settings** Nursing care facilities (NAICS 6231) and hospitals (NAICS 622)
**Source data** US Government work, 17 U.S.C. §105

### Disclosure

The author distributes a powered lateral-transfer system. That is a reason to state plainly what this analysis can and cannot show, and to report the result whichever way it falls. No product is named or evaluated.

### What this analysis can and cannot show

- **Task named is not task frequency.** The data counts injuries. It does not count how often each task is performed, so it cannot say which task is most dangerous per transfer. A task can account for few injuries because it is rare, or because it is safe.
- **Many narratives name no task.** They are reported as "transfer, type not stated" or "handling, task not stated", not assigned a guess.
- **The handling rule was validated on nursing-facility narratives.** It is applied unchanged to hospitals here; its accuracy there has not yet been measured.

## 1. Environment and data

In [1]:
!pip install -q duckdb pandas requests
import os, re, duckdb, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
PATH = 'data/osha_cases.csv'
if not os.path.exists(PATH):
    r = requests.get(CASES_URL, headers=HDRS, timeout=1800, stream=True); r.raise_for_status()
    with open(PATH, 'wb') as f:
        for ch in r.iter_content(1 << 20): f.write(ch)
con = duckdb.connect(':memory:')
con.execute(f"CREATE TABLE raw AS SELECT * FROM read_csv_auto('{PATH}', all_varchar=true, ignore_errors=true, sample_size=-1)")
cols = [r[0] for r in con.execute('DESCRIBE raw').fetchall()]
low = {c.lower(): c for c in cols}
def col(c): return low[c.lower()]
nar = "lower(concat_ws(' ', " + ', '.join(f'"{col(c)}"' for c in
      ['new_nar_before_incident', 'new_nar_what_happened', 'new_nar_injury_illness', 'new_nar_object_substance']) + '))'
df = con.execute(f'''SELECT {nar} AS narr, "{col('naics_code')}" AS naics,
       CASE WHEN TRY_CAST("{col('dafw_num_away')}" AS INT) >= 0 THEN TRY_CAST("{col('dafw_num_away')}" AS INT) END AS dafw
     FROM raw WHERE "{col('naics_code')}" LIKE '6231%' OR "{col('naics_code')}" LIKE '622%' ''').df()
df['setting'] = np.where(df.naics.str.startswith('6231'), 'nursing facility', 'hospital')
print(df.setting.value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

setting
hospital            181690
nursing facility     25860
Name: count, dtype: int64


## 2. The frozen handling rule (unchanged)

In [2]:
PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
n = df.narr.fillna('')
df['handling'] = (n.str.contains(PERSON, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
print(df.groupby('setting').handling.agg(['sum', 'mean']).round(3))
print('nursing-facility handling should equal 5,654')

                    sum   mean
setting                       
hospital          20685  0.114
nursing facility   5654  0.219
nursing-facility handling should equal 5,654


## 3. Task type

Each handling case is assigned the first task, in this order, that its narrative names:

1. **Lateral transfer between surfaces**: moving a person who is lying down from one surface to another: bed to bed, bed to operating, procedure, exam, or imaging table, table to bed, and bed to or from a stretcher or gurney. A slide board used to move a seated person from bed to wheelchair, or moving a seated person from a wheelchair onto a stretcher, is a seated transfer, not a lateral one. A table or stretcher named without a transfer to or from it (for example, boosting a person who is already on an exam table) is not counted as lateral.
2. **Recovery from the floor**: lifting or helping a person up off the floor.
3. **Catching or supporting a falling person.**
4. **Seated or standing transfer**: bed, chair, wheelchair, commode, or toilet to another of these; sit-to-stand; stand-pivot.
5. **Repositioning in bed**: pulling up, boosting, turning, or rolling a person in bed.
6. **Ambulation**: walking a person.
7. **Transfer, type not stated**: the narrative says "transfer" without saying from or to what.
8. **Handling, task not stated.**

The order puts the most specific descriptions first. A case naming both a fall and a seated transfer is counted as catching, because that is the event the narrative describes. Spot checks follow.

In [3]:
TASKS = [
 ('lateral transfer between surfaces',
  r"\b(?:lateral(?:ly)? (?:transfer\w*|slid\w*|mov\w*)|bed to bed|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:operating|or|surgical|procedure|exam|examination|ct|cat scan|mri|x-?ray|imaging|radiology|cath\w*|ir|endoscopy|dialysis) (?:room )?(?:table|bed)|"
  r"(?:operating|or|surgical|procedure|exam|ct|mri|x-?ray|imaging) table to|"
  r"(?:bed|table|stretcher|gurney|cart|litter|cot) to (?:a |an |the |another |new |hospital |ems |shower |her |his )?(?:bed|table|stretcher|gurney|cart|litter|cot)|"
  r"(?:to|from|onto|off|off of|on to|over to) (?:the |a |an |his |her )?(?:\w+ )?(?:stretcher|gurney|litter|cot)\b|(?:stretcher|gurney|litter|cot) to|"
  r"slid\w* (?:the )?(?:resident|patient|pt|him|her) (?:over|across|onto|from))\b"),
 ('recovery from the floor',
  r"\b(?:off (?:of )?the (?:floor|ground)|from the (?:floor|ground)|up off|(?:found|was|lying) on the floor|lift\w* (?:\w+ ){0,2}(?:from|off) (?:the )?floor)\b"),
 ('catching or supporting a falling person',
  r"\b(?:catch\w*|caught (?:the |a )?(?:resident|patient|him|her)|began to fall|started to fall|was falling|"
  r"prevent\w* (?:\w+ ){0,2}from falling|keep \w+ from falling|lower\w* (?:\w+ ){0,2}to the (?:floor|ground)|(?:legs|knees) (?:gave out|buckled))\b"),
 ('seated or standing transfer',
  r"\b(?:(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair) to (?:a |the )?(?:bed|chair|recliner|wheel ?chair|w/?c|commode|toilet|shower chair|geri ?chair)|"
  r"sit[- ]to[- ]stand|stand[- ]pivot|pivot\w*|into (?:the |a |her |his )?(?:wheel ?chair|chair|recliner|w/?c)|out of (?:the |a |her |his )?(?:wheel ?chair|chair|bed|recliner)|"
  r"on(?:to)? (?:the )?(?:toilet|commode)|off (?:the )?(?:toilet|commode)|(?:to|help\w* \w+) stand)\b"),
 ('repositioning in bed',
  r"\b(?:pull\w* (?:\w+ ){0,3}up (?:in|to)|boost\w*|reposition\w*|turn\w* (?:\w+ ){0,3}(?:in bed|over|on (?:her|his|their|the) side)|"
  r"roll\w* (?:the |a )?(?:resident|patient|pt|him|her)|draw ?sheet|turning sheet|up in bed|head of (?:the )?bed)\b"),
 ('ambulation', r"\b(?:ambulat\w*|walking (?:the |a |with (?:the |a )?)?(?:resident|patient|pt))\b"),
 ('transfer, type not stated', r"\btransfer\w*\b"),
]
SEATED_TO_STRETCHER = r"\b(?:wheel ?chair|chair|w/?c|recliner|geri ?chair) to (?:a |an |the )?(?:\w+ )?(?:stretcher|gurney|cot|table)\b"
def task(s):
    if re.search(SEATED_TO_STRETCHER, s): return 'seated or standing transfer'
    for name, p in TASKS:
        if re.search(p, s): return name
    return 'handling, task not stated'
h = df[df.handling].copy()
h['task'] = h.narr.fillna('').map(task)
order = [t for t, _ in TASKS] + ['handling, task not stated']
tab = h.groupby(['task', 'setting']).size().unstack(1).reindex(order).fillna(0).astype(int)
for s_ in tab.columns: tab[f'{s_} share'] = (tab[s_] / tab[s_].sum()).round(3)
tab

setting,hospital,nursing facility,hospital share,nursing facility share
task,,,,
lateral transfer between surfaces,1726,22,0.083,0.004
recovery from the floor,280,145,0.014,0.026
catching or supporting a falling person,780,230,0.038,0.041
seated or standing transfer,1942,788,0.094,0.139
repositioning in bed,5347,1231,0.258,0.218
ambulation,304,24,0.015,0.004
"transfer, type not stated",2951,1482,0.143,0.262
"handling, task not stated",7355,1732,0.356,0.306


### Spot check: lateral transfers

In [4]:
for s_ in ['hospital', 'nursing facility']:
    x = h[(h.task == 'lateral transfer between surfaces') & (h.setting == s_)]
    print(f'===== {s_} ({len(x):,}) =====')
    for t in x.narr.sample(min(15, len(x)), random_state=1): print(' -', t[:220])

===== hospital (1,726) =====
 - moving heavy patient from hospital [redacted] mri tech was moving a heavy patient from their hospital bed to the mri exam table mri tech felt some strain in lower back but thought nothing of it around [redacted] began fe
 - transferring patient from ct table to bed while transferring patient using a slide board felt pain in the upper arm and shoulder area right arm and shoulder strain weight of patient
 - turning and repositioning patient. transferred to or table pulled something in right shoulder neck and lower back soreness pain multiple body parts right patient
 - talking with the patient patient was in wheelchair. prepared for full assist transfer to stretcher had upper body weight partner had legs felt strain to muscle oft side of back strain to left lower back and left shoulder
 - cleaning patient then was going to transfer patient [redacted] felt low back pain and back spasms after transferring patient with [redacted] to stretcher after cleaning p

### Spot check: other task types (nursing facilities)

In [5]:
for t_ in order[1:]:
    x = h[(h.task == t_) & (h.setting == 'nursing facility')]
    print(f'===== {t_} ({len(x):,}) =====')
    for t in x.narr.sample(min(5, len(x)), random_state=1): print(' -', t[:180])

===== recovery from the floor (145) =====
 - resident care 3 staff were transferring a resident to bed with a transfer sheet from the floor. staff landed in bed with the resident and strained their back. strained upper back n
 - on break when another staff person yelled that a resident was on the floor. we tried to use a hoyer lift the lift was not working she used her left arm and hand to protect the resi
 - assisted resident off of toilet employee was assisting a resident to stand up off the toilet without the use of a mechanical lifting device. strain to shoulder liftine resident
 - answering resident 's call light concussion. went to turn off light foot got caught in telephone cord fell hit head on closet handle and back of head was on the floor and hit left 
 - assisting the nurse. the employee was assisting the nurse with transferring the resident off the floor. employee states she hurt her shoulder and neck. strain right trap neck no ob
===== catching or supporting a falling per

## 4. Severity by task type

In [6]:
sev = (h.groupby(['setting', 'task']).agg(cases=('narr', 'size'), mean_days_away=('dafw', 'mean'),
        median_days_away=('dafw', 'median'), over_90_days=('dafw', lambda s: int((s > 90).sum()))).round(1))
sev

cases  mean_days_away  median_days_away  over_90_days
setting          task                                                                                          
hospital         ambulation                                 304            18.6               0.0            23
                 catching or supporting a falling person    780            14.9               0.0            35
                 handling, task not stated                 7355            15.0               0.0           439
                 lateral transfer between surfaces         1726            16.5               0.0           105
                 recovery from the floor                    280            13.3               0.0            13
                 repositioning in bed                      5347            16.4               0.0           341
                 seated or standing transfer               1942            16.9               0.0           127
                 transfer, type not stated                 2951            11.6               0.0           128
nursing facility ambulation                                  24             7.6               0.0             1
                 catching or supporting a falling person    230             6.7               0.0             7
                 handling, task not stated                 1732            10.4               0.0            66
                 lateral transfer between surfaces           22             1.8               0.0             0
                 recovery from the floor                    145             9.6               0.0             6
                 repositioning in bed                      1231             9.6               0.0            36
                 seated or standing transfer                788            11.2               0.0            35
                 transfer, type not stated                 1482            11.1               0.0            64

## 5. Validation sample

The task patterns are keyword rules. The cell writes a blind sample of 150 handling cases, 75 per setting, drawn at random, narrative only, for a reviewer to assign one of the eight task types. Lateral transfers are also oversampled separately (up to 50) because they are rare and their count is the figure most likely to be cited.

In [7]:
a = pd.concat([g.sample(min(75, len(g)), random_state=2026) for _, g in h.groupby('setting')])
lat = h[h.task == 'lateral transfer between surfaces']
b = lat.drop(a.index, errors='ignore').sample(min(50, len(lat)), random_state=2026)
val = pd.concat([a.assign(sample='random'), b.assign(sample='lateral oversample')]).sample(frac=1, random_state=7)
val[['setting', 'sample', 'narr']].assign(label='').reset_index(names='row').to_csv('derived/task-validation-sample.csv', index=False)
print(len(val), 'cases written')

200 cases written


## 6. Export

In [8]:
tab.to_csv('derived/task-by-setting.csv'); sev.to_csv('derived/severity-by-task.csv')
with open('derived/classification-rule.txt', 'w') as f:
    for name, p in TASKS: f.write(f'{name}:\n  {p}\n')
    f.write('Assigned in the order above; first match wins. Otherwise: handling, task not stated.\n')
print(sorted(os.listdir('derived')))

['classification-rule.txt', 'severity-by-task.csv', 'task-by-setting.csv', 'task-validation-sample.csv']


In [9]:
!zip -r task-final.zip derived

  adding: derived/ (stored 0%)
  adding: derived/task-by-setting.csv (deflated 40%)
  adding: derived/severity-by-task.csv (deflated 61%)
  adding: derived/classification-rule.txt (deflated 61%)
  adding: derived/task-validation-sample.csv (deflated 72%)
